In [ ]:
-- ============================================================
-- mart_trial_activation.sql
-- Mart layer: tracks which organisations achieved Trial Activation
-- One row per organisation
-- Activation = completing ALL 5 trial goals
-- ============================================================

WITH trial_goals AS (
    SELECT * FROM {{ ref('mart_trial_goals') }}
    -- Note: replace {{ ref('mart_trial_goals') }} with mart_trial_goals
    -- if not using dbt
),

activation AS (
    SELECT
        ORGANIZATION_ID,
        is_converted,

        -- Individual goals
        goal_1_shift_created,
        goal_2_shift_approved,
        goal_3_template_applied,
        goal_4_open_shift_request,
        goal_5_punched_in,

        -- Total goals completed
        (
            goal_1_shift_created +
            goal_2_shift_approved +
            goal_3_template_applied +
            goal_4_open_shift_request +
            goal_5_punched_in
        )                                AS total_goals_completed,

        -- Activation flag
        -- An org is activated when ALL 5 goals are completed
        CASE WHEN (
            goal_1_shift_created = 1 AND
            goal_2_shift_approved = 1 AND
            goal_3_template_applied = 1 AND
            goal_4_open_shift_request = 1 AND
            goal_5_punched_in = 1
        ) THEN 1 ELSE 0 END             AS is_activated,

        -- Activation tier based on goals completed
        CASE
            WHEN (
                goal_1_shift_created +
                goal_2_shift_approved +
                goal_3_template_applied +
                goal_4_open_shift_request +
                goal_5_punched_in
            ) = 5 THEN 'Fully Activated'
            WHEN (
                goal_1_shift_created +
                goal_2_shift_approved +
                goal_3_template_applied +
                goal_4_open_shift_request +
                goal_5_punched_in
            ) >= 3 THEN 'Partially Activated'
            WHEN (
                goal_1_shift_created +
                goal_2_shift_approved +
                goal_3_template_applied +
                goal_4_open_shift_request +
                goal_5_punched_in
            ) >= 1 THEN 'Minimal Engagement'
            ELSE 'No Engagement'
        END                             AS activation_tier

    FROM trial_goals
)

SELECT * FROM activation